# LeukoQ: Quantum Machine Learning for Leukemia Detection

**Objective**: Detect early-stage leukemia (ALL vs AML) using a hybrid Quantum-Classical pipeline.

**Authors**: Adiba (Project Lead), Lian Mollick (Supervisor)
**Affiliation**: Viqarunnisa Noon School & College, ASL Research Group

---

## 📊 DATASET ATTRIBUTION & SCIENTIFIC CREDITS
This research would not be possible without the open sharing of clinical data by the following researchers:

1. **Todd R. Golub et al. (Science, 1999)**: For the foundational Leukemia gene expression dataset (OpenML #1104). This work pioneered the field of cancer genomics.
2. **The MedMNIST/BloodMNIST Team (Yang et al.)**: For the BloodMNIST microscopy library (17,092 images) used for training cell image classifiers.
3. **Kaggle & OpenML Communities**: For providing the public benchmarks and pediatric hematology records used in this research.

*Technical assistance provided by Google Gemini and Anthropic Claude.*

---

## 1. Installation
We use `Qiskit` for variational circuits and `PennyLane` for quantum neural networks.

In [ ]:
!pip install -q qiskit qiskit-machine-learning qiskit-aer pennylane scikit-learn shap matplotlib seaborn

## 2. Data Loading (OpenML Leukemia)
The dataset contains 72 patients and 7,129 gene expression features.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer, StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA

# Load data
print("Fetching OpenML leukemia dataset...")
data = fetch_openml(name="leukemia", version=1, as_frame=False, parser="auto")
X = data.data.astype(np.float64)
y_raw = np.array(data.target)
classes = sorted(np.unique(y_raw))
y = (y_raw == classes[1]).astype(int) # ALL=0, AML=1
feature_names = data.feature_names if hasattr(data, 'feature_names') else [f'gene_{i}' for i in range(X.shape[1])]

print(f"Dataset loaded: {X.shape[0]} samples, {X.shape[1]} genes")

## 3. Preprocessing Pipeline
We use ANOVA F-test to select the top 64 genes and then PCA for quantum feature reduction.

In [ ]:
# Preprocess for Classical
qt = QuantileTransformer(output_distribution="normal", random_state=42)
X_norm = qt.fit_transform(X)

selector = SelectKBest(score_func=f_classif, k=64)
X_sel = selector.fit_transform(X_norm, y)

scaler = StandardScaler()
X_cl = scaler.fit_transform(X_sel)

X_train, X_test, y_train, y_test = train_test_split(X_cl, y, test_size=0.2, stratify=y, random_state=42)

# Preprocess for Quantum (4 qubits)
pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_sel)
angle_scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
X_q = angle_scaler.fit_transform(X_pca)

X_q_train, X_q_test, _, _ = train_test_split(X_q, y, test_size=0.2, stratify=y, random_state=42)

## 4. Model Training
Training a Hybrid Quantum-Classical Ensemble.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Classical Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
print("Classification Results:")
print(classification_report(y_test, rf_preds))

## 5. Explainability (SHAP)
Visualizing gene importance for the predictions.

In [ ]:
import shap
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)
selected_gene_names = [feature_names[i] for i in selector.get_support(indices=True)]
shap.summary_plot(shap_values[1], X_test, feature_names=selected_gene_names)